# FreshCheck C+CVAT Runner

Notebook นี้แยกเฉพาะ workflow `C+CVAT`

ใช้เมื่อ:
- XML อยู่ที่ `/content/drive/MyDrive/FreshCheck/data/merged_annotations.xml`
- รูปจริงอยู่ใต้ `/content/drive/MyDrive/FreshCheck/data/new_test/FR`, `HF`, `SP`
- ต้องการรัน mask-guided crop + tuned classification แยกจาก notebook หลัก


In [ ]:
# 1) Clone / Update repository
REPO_URL = "https://github.com/techasit239/DADS7202_PigPicture.git"
BRANCH = "main"
REPO_DIR = "/content/DADS7202_PigPicture"

import subprocess
from pathlib import Path

repo_path = Path(REPO_DIR)
if repo_path.exists():
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo_path), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_path)], check=True)

%cd /content/DADS7202_PigPicture


In [ ]:
# 2) Mount Google Drive and set paths
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/FreshCheck')
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'

CPLUS_CVAT_IMAGE_DIR = '/content/drive/MyDrive/FreshCheck/data/new_test'
CPLUS_CVAT_XML_PATH = '/content/drive/MyDrive/FreshCheck/data/merged_annotations.xml'

CVAT_ROOT = '/content/drive/MyDrive/FreshCheck/artifacts/phase2_cplus_cvat'
CVAT_MANIFEST = f'{CVAT_ROOT}/thai_test_set.csv'
CVAT_CROP_MANIFEST = f'{CVAT_ROOT}/crop_manifest.csv'
CVAT_EXPERIMENTS = '/content/drive/MyDrive/FreshCheck/artifacts/experiments_cplus_cvat'

CPLUS_CVAT_EFF_OUT = '/content/drive/MyDrive/FreshCheck/artifacts/exp_cplus_cvat_efficientnet'
CPLUS_CVAT_SWIN_OUT = '/content/drive/MyDrive/FreshCheck/artifacts/exp_cplus_cvat_swin'
CPLUS_CVAT_TARGET_EVAL = '/content/drive/MyDrive/FreshCheck/artifacts/exp_cplus_cvat_eval_target'
CPLUS_CVAT_SOURCE_EVAL = '/content/drive/MyDrive/FreshCheck/artifacts/exp_cplus_cvat_eval_source'

print("image dir :", CPLUS_CVAT_IMAGE_DIR)
print("xml path  :", CPLUS_CVAT_XML_PATH)
print("CVAT root :", CVAT_ROOT)


In [ ]:
# 3) Install dependencies
%cd /content/DADS7202_PigPicture
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt


In [ ]:
# 4) Remove old C+CVAT outputs
import shutil
from pathlib import Path

for p in [
    CVAT_ROOT,
    CVAT_EXPERIMENTS,
    CPLUS_CVAT_EFF_OUT,
    CPLUS_CVAT_SWIN_OUT,
    CPLUS_CVAT_TARGET_EVAL,
    CPLUS_CVAT_SOURCE_EVAL,
]:
    path = Path(p)
    if path.exists():
        shutil.rmtree(path)
        print("deleted", path)
print("cleanup complete")


In [ ]:
# 5) Prepare CVAT manifest and GT masks
!python run_freshcheck.py prepare-cvat \
  --thai-data-dir "$CPLUS_CVAT_IMAGE_DIR" \
  --cvat-xml-path "$CPLUS_CVAT_XML_PATH" \
  --output-dir "$CVAT_ROOT" 


In [ ]:
# 6) Build cropped pork images
!python run_freshcheck.py crop-from-masks \
  --csv "$CVAT_MANIFEST" \
  --output-dir "$CVAT_ROOT" \
  --filename crop_manifest.csv \
  --margin-ratio 0.08 \
  --min-pixels 25


In [ ]:
# 7) Prepare experiments using cropped target images
!python run_freshcheck.py prepare-experiments \
  --source-csv /content/drive/MyDrive/FreshCheck/artifacts/manifests/source_train_manifest.csv \
  --target-csv "$CVAT_CROP_MANIFEST" \
  --output-dir "$CVAT_EXPERIMENTS" \
  --source-fit-ratio 0.8 \
  --target-fit-ratio 0.7 \
  --train-ratio-within-fit 0.8 \
  --seed 42


In [ ]:
# 8) Check class distributions before training
import pandas as pd

print("CVAT manifest")
print(pd.read_csv(CVAT_MANIFEST)["class"].value_counts())

print("\nCrop manifest")
print(pd.read_csv(CVAT_CROP_MANIFEST)["class"].value_counts())

print("\nTarget holdout")
print(pd.read_csv(f"{CVAT_EXPERIMENTS}/target_holdout.csv")["class"].value_counts())


In [ ]:
# 9) Train EfficientNet-B0
!python run_freshcheck.py train \
  --train-csv "$CVAT_EXPERIMENTS/exp_c_train_rebalanced.csv" \
  --val-csv "$CVAT_EXPERIMENTS/exp_c_val.csv" \
  --output-dir "$CPLUS_CVAT_EFF_OUT" \
  --models efficientnet_b0 \
  --epochs 22 \
  --patience 6 \
  --lr 0.0003 \
  --weight-decay 0.005 \
  --dropout 0.25 \
  --img-size 224 \
  --batch-size 32 \
  --num-workers 2


In [ ]:
# 10) Train Swin-T
!python run_freshcheck.py train \
  --train-csv "$CVAT_EXPERIMENTS/exp_c_train_rebalanced.csv" \
  --val-csv "$CVAT_EXPERIMENTS/exp_c_val.csv" \
  --output-dir "$CPLUS_CVAT_SWIN_OUT" \
  --models swin_t \
  --epochs 14 \
  --patience 4 \
  --lr 0.00008 \
  --weight-decay 0.01 \
  --dropout 0.30 \
  --img-size 224 \
  --batch-size 16 \
  --num-workers 2


In [ ]:
# 11) Evaluate on target holdout
!python run_freshcheck.py evaluate \
  --csv "$CVAT_EXPERIMENTS/target_holdout.csv" \
  --output-dir "$CPLUS_CVAT_TARGET_EVAL" \
  --models efficientnet_b0 swin_t \
  --checkpoint-paths \
    efficientnet_b0="$CPLUS_CVAT_EFF_OUT/checkpoints/efficientnet_b0_best.pt" \
    swin_t="$CPLUS_CVAT_SWIN_OUT/checkpoints/swin_t_best.pt" \
  --batch-size 16 \
  --num-workers 2


In [ ]:
# 12) Evaluate on source holdout
!python run_freshcheck.py evaluate \
  --csv "$CVAT_EXPERIMENTS/source_holdout.csv" \
  --output-dir "$CPLUS_CVAT_SOURCE_EVAL" \
  --models efficientnet_b0 swin_t \
  --checkpoint-paths \
    efficientnet_b0="$CPLUS_CVAT_EFF_OUT/checkpoints/efficientnet_b0_best.pt" \
    swin_t="$CPLUS_CVAT_SWIN_OUT/checkpoints/swin_t_best.pt" \
  --batch-size 16 \
  --num-workers 2


In [ ]:
# 13) Summarize evaluation results
import json
from pathlib import Path

def read_eval(path):
    path = Path(path)
    if not path.exists():
        print(f"[MISSING] {path}")
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    r = data.get("results", {})
    print(f"\n=== {path.stem} ===")
    print(f"accuracy        : {r.get('accuracy', 0):.4f}")
    print(f"macro_precision : {r.get('macro_precision', 0):.4f}")
    print(f"macro_recall    : {r.get('macro_recall', 0):.4f}")
    print(f"macro_f1        : {r.get('macro_f1', 0):.4f}")
    print(f"loss            : {r.get('loss', 0):.4f}")
    cm = data.get("confusion_matrix", {})
    print("confusion labels:", cm.get("labels"))
    print("confusion matrix:")
    for row in cm.get("matrix", []):
        print(row)

for eval_path in [
    Path(CPLUS_CVAT_TARGET_EVAL) / 'efficientnet_b0_eval.json',
    Path(CPLUS_CVAT_TARGET_EVAL) / 'swin_t_eval.json',
    Path(CPLUS_CVAT_SOURCE_EVAL) / 'efficientnet_b0_eval.json',
    Path(CPLUS_CVAT_SOURCE_EVAL) / 'swin_t_eval.json',
]:
    read_eval(eval_path)
